# Clean + combine training data

## Train from a CSV. Call APIs when you predict.

| Source | Put in the training CSV? | Why |
|---|---|---|
| **LUCAS 2018** chemistry | Yes | Labels + nutrients, Europe |
| **LUCAS 2015** clay/sand/silt | Yes, join on point ID | Texture only where the same point exists |
| **WoSIS** orgc, pH, clay, sand, silt, N | Yes | Global (including Oceania) |
| **GEE Sentinel NDVI now** | No (not for every row) | Satellite *today* is not the 2018 lab sample |
| **OpenWeather current** | No | Weather *today* does not match sample dates |
| **GEE WorldClim** (optional later) | Yes, as extra columns | Long-run climate at that lat/lon — fair training feature |

You **cannot** “train with the API itself” the way people train on a CSV. You either:

1. **Pull features once**, save them as columns, train on the file (good), or
2. Call APIs **at inference** for a farm point (GEE + OpenWeather) after the model is trained.

Do **not** add SoilGrids / extra satellite vendors yet. GEE already has Sentinel + climate. SoilGrids duplicates WoSIS-style soil maps.

Ignored on purpose: other WoSIS TSVs, LUCAS shapefiles, erosion/ORG extras, bulk density.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from build_training_csv import main, OUT_CSV

sns.set_theme(style="whitegrid")
path = main()
df = pd.read_csv(path)
df.head()

In [ ]:
print(df.shape)
print(df["source"].value_counts())
print("\nDtypes:\n", df.dtypes)
print("\nMissing %:\n", (df.isna().mean() * 100).round(1))
df[["ph_h2o", "oc_gkg", "n_gkg", "clay_pct", "acidic", "low_oc"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df, x="ph_h2o", hue="source", bins=30, ax=axes[0], element="step")
sns.histplot(df[df["oc_gkg"] < 80], x="oc_gkg", hue="source", bins=30, ax=axes[1], element="step")
sns.scatterplot(
    data=df.sample(min(6000, len(df)), random_state=0),
    x="ph_h2o", y="oc_gkg", hue="source", ax=axes[2], alpha=0.35, s=10,
)
axes[2].set_ylim(0, 80)
plt.tight_layout()
plt.show()
print("Saved:", path.resolve())

## Optional: climate columns from GEE (not OpenWeather)

WorldClim is a **climatology** (typical temperature and rainfall at that point). That is a valid extra feature for every training row.

This cell samples **400 random rows** only. Sampling all ~100k+ points in a notebook will be slow and can hit quota. If this works, we can export a full GEE table later.

Skip this cell until soil CSV looks right.

In [ ]:
import os
import ee
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
GEE_PROJECT_ID = os.getenv("GEE_PROJECT_ID", "valued-aquifer-507001-v2")
ee.Initialize(project=GEE_PROJECT_ID)

sample = df.dropna(subset=["latitude", "longitude"]).sample(400, random_state=0).copy()
features = []
for i, row in sample.iterrows():
    geom = ee.Geometry.Point([float(row["longitude"]), float(row["latitude"])])
    features.append(ee.Feature(geom, {"ix": int(i)}))
fc = ee.FeatureCollection(features)

clim = ee.Image("WORLDCLIM/V1/BIO").select(["bio01", "bio12"], ["tmean_x10", "precip_mm"])
sampled = clim.sampleRegions(collection=fc, scale=1000, geometries=False).getInfo()

rows = []
for f in sampled["features"]:
    p = f["properties"]
    rows.append(p)
clim_df = pd.DataFrame(rows).set_index("ix")
# bio01 is temperature * 10 (°C)
clim_df["tmean_c"] = clim_df["tmean_x10"] / 10.0

sample = sample.join(clim_df[["tmean_c", "precip_mm"]])
out_sample = Path("data") / "agrishield_training_gee_climate_sample.csv"
sample.to_csv(out_sample, index=False)
print(sample[["source", "ph_h2o", "tmean_c", "precip_mm"]].head())
print("Wrote", out_sample)